In [1]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import Markdown, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.iolib.summary2 import summary_col

# Не показывать Warning
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# Настройки оформления графиков
plt.rcParams['figure.figsize'] = (8.5, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

CONST_NAMES = ('const', 'Intercept')


def fmt(x, decimals=3):
    """Число с заданным числом знаков; оценка, которая округляется до нуля, — с тремя значащими цифрами."""
    if pd.isna(x):
        return ''
    text = f'{x:.{decimals}f}'
    if float(text) == 0:
        return f'{x:.3g}' if x != 0 else f'{0:.{decimals}f}'
    return text


def fmt_p(p, decimals=3):
    """P-значение: меньше 0.001 печатается как «<0.001»."""
    return '<0.001' if p < 0.001 else f'{p:.{decimals}f}'


def errors_text(res, case='nom'):
    """Название типа стандартных ошибок подгонки: 'nom' — «неробастные», 'dat' — «по неробастным …»."""
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    if cov_type == 'nonrobust':
        return {'nom': 'неробастные стандартные ошибки',
                'dat': 'по неробастным стандартным ошибкам'}[case]
    return {'nom': f'робастные стандартные ошибки, {cov_type}',
            'dat': f'по робастным стандартным ошибкам {cov_type}'}[case]


def fit_ols(formula, data, robust=False, **kwargs):
    """МНК-подгонка; robust=True — робастные ошибки HC3. Всегда t- и F-распределения (use_t=True)."""
    cov_type = 'HC3' if robust else 'nonrobust'
    return ols(formula, data=data).fit(cov_type=cov_type, use_t=True, **kwargs)


def show_table(df, index=True, align=None):
    """Таблица pandas в Markdown; числа должны быть заранее превращены в строки через fmt."""
    display(Markdown('\n\n' + df.to_markdown(index=index, colalign=align, disable_numparse=True) + '\n\n'))


def nobs_text(res):
    """Строка «Модель была подогнана по N наблюдениям»."""
    n = int(res.nobs)
    word = 'наблюдению' if n % 10 == 1 and n % 100 != 11 else 'наблюдениям'
    return f'Модель была подогнана по {n} {word}.'


def show_coefs(res, decimals=3, show_nobs=True):
    """Оценки коэффициентов: вертикальная таблица «Переменная | Оценка» и число наблюдений."""
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    column = 'Оценка' if cov_type == 'nonrobust' else f'Оценка ({cov_type})'
    table = pd.DataFrame({'Переменная': res.params.index,
                          column: [fmt(b, decimals) for b in res.params]})
    text = table.to_markdown(index=False, colalign=('left', 'right'), disable_numparse=True)
    if show_nobs:
        text += '\n\n' + nobs_text(res)
    display(Markdown('\n\n' + text + '\n\n'))


def display_answer(text, use_callout=True, theme='note', title='Критическое значение'):
    """Ответ в рамке (callout): theme='note' — число, 'tip' — вывод, 'important' — условие."""
    if use_callout:
        head = f'title="{title}" ' if title else ''
        output_md = f'\n\n::: {{.callout-{theme} {head}icon="true"}}\n{text}\n:::\n\n'
    elif title:
        output_md = f'\n\n**{title}:** {text}\n\n'
    else:
        output_md = f'\n\n{text}\n\n'
    display(Markdown(output_md))


def show_conditions(level, errors=None, hypothesis=None, extra=()):
    """Блок «Условия теста»: уровень значимости (доля или список долей), тип ошибок, гипотеза формулой."""
    levels = level if isinstance(level, (list, tuple)) else [level]
    items = ['уровень значимости: **' + ' и '.join(f'{a * 100:g}%' for a in levels) + '**']
    if errors:
        items.append(f'стандартные ошибки: **{errors}**')
    hypotheses = [hypothesis] if isinstance(hypothesis, str) else list(hypothesis or [])
    if len(hypotheses) == 1:
        items.append(f'гипотеза: {hypotheses[0]}')
    else:
        items += [f'гипотеза {i}: {h}' for i, h in enumerate(hypotheses, start=1)]
    items += list(extra)
    display_answer('\n'.join(f'- {item}' for item in items), theme='important', title='Условия теста')


def signif_stars(p):
    """Коды значимости: *** p<0.01, ** p<0.05, * p<0.1."""
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''


SIGNIF_LEGEND = '*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$'


def print_regression_summary(res, decimals=3, show_signif=True):
    """Таблица t-теста: Coef., Std.Err., t-value, p-value, Signif. Заголовок называет тип ошибок."""
    title = f'**Результаты $t$-теста для коэффициентов ({errors_text(res)}):**'
    table = pd.DataFrame({
        'Coef.': [fmt(b, decimals) for b in res.params],
        'Std.Err.': [fmt(s, decimals) for s in res.bse],
        't-value': [fmt(t, decimals) for t in res.tvalues],
        'p-value': [fmt_p(p, decimals) for p in res.pvalues],
    }, index=res.params.index)
    legend = ''
    if show_signif:
        table['Signif.'] = ['' if name in CONST_NAMES else signif_stars(p)
                            for name, p in res.pvalues.items()]
        legend = '\n\n' + SIGNIF_LEGEND
    align = ('left',) + ('right',) * 4 + (('left',) if show_signif else ())
    display(Markdown(f'\n\n{title}\n\n' + table.to_markdown(colalign=align, disable_numparse=True) + legend + '\n\n'))


def print_significant_coeffs(res, sign_level=0.05, theme='note', title=None):
    """Рамка со списком коэффициентов, значимых на уровне sign_level (без константы)."""
    names = [name for name, p in res.pvalues.items() if p < sign_level and name not in CONST_NAMES]
    if title is None:
        title = f'Коэффициенты, значимые на уровне {sign_level * 100:g}% ({errors_text(res, "dat")})'
    text = ', '.join(f'`{name}`' for name in names) if names else 'Значимых коэффициентов не выявлено.'
    display_answer(text, theme=theme, title=title)


def show_models(models, names=None, decimals=3, regressor_order=(), errors_note=None, info=None):
    """Сравнение регрессий (summary_col): строка Dep. Variable, коды значимости, пояснение про скобки.

    info — словарь {название строки: функция от подгонки} для дополнительных строк (N, F и т. п.).
    """
    table = summary_col(models, model_names=names, stars=True, float_format=f'%0.{decimals}f',
                        regressor_order=list(regressor_order), info_dict=info or {}).tables[0]
    top = pd.DataFrame({col: [m.model.endog_names] for col, m in zip(table.columns, models)},
                       index=['Dep. Variable'])
    table = pd.concat([top, table])
    if errors_note is None:
        errors_note = errors_text(models[0])
    note = f'{SIGNIF_LEGEND}. В скобках — {errors_note}.'
    display(Markdown('\n\n' + table.to_markdown(disable_numparse=True) + '\n\n' + note + '\n\n'))


def show_ci(res, level=0.95, decimals=3):
    """Доверительные интервалы: «Переменная | Нижняя граница | Верхняя граница», заголовок над таблицей."""
    ci = res.conf_int(alpha=1 - level)
    table = pd.DataFrame({'Переменная': ci.index,
                          'Нижняя граница': [fmt(x, decimals) for x in ci[0]],
                          'Верхняя граница': [fmt(x, decimals) for x in ci[1]]})
    title = f'**{level * 100:g}%-е доверительные интервалы ({errors_text(res)}):**'
    display(Markdown(f'\n\n{title}\n\n'
                     + table.to_markdown(index=False, colalign=('left', 'right', 'right'), disable_numparse=True) + '\n\n'))


# Загрузка данных
sleep75 = pd.read_csv('../datasets/sleep75.csv')
wage2 = pd.read_csv('../datasets/wage2.csv')
Labour = pd.read_csv('../datasets/Labour.csv')
Electricity = pd.read_csv('../datasets/Electricity.csv')


def direction(b, ending='ется'):
    """Глагол для интерпретации по знаку оценки: «увеличивается» / «уменьшается»."""
    return ('увеличива' if b > 0 else 'уменьша') + ending

::: {.callout-important title="Важно"}
- Логарифм везде натуральный; константа включается в модель по умолчанию.
- Данные лежат в папке `datasets/`, описание переменных — в `datasets/data-description.md`.
- Оценки для `sleep75` округляются до 2-х десятичных знаков, для остальных наборов данных — до 3-х.
- Ответы считаются по неокруглённым оценкам.
- Коэффициенты подписаны так, как их печатает statsmodels: `Intercept` — константа, `np.log(x)` — коэффициент при $\log x$.
- Образец интерпретации дан по одному на каждый тип модели: уровень–уровень, уровень–лог, лог–уровень, лог–лог и бинарная переменная. В остальных задачах приведены только оценки.
:::

# Модели уровень–уровень

## sleep equation #1

Для набора данных `sleep75` рассмотрим регрессию **sleep на totwrk, male**.

Объясняющие переменные: `totwrk` и `male`; контрольных нет.

Переменные: `sleep` — ночной сон, минут в неделю; `totwrk` — работа, минут в неделю; `male` = 1 для мужчин.

Запишите спецификацию модели.

Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

Ответ:

In [2]:
#| output: asis
res = fit_ols('sleep ~ totwrk + male', sleep75)
show_coefs(res, decimals=2)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |  3573.20 |
| totwrk       |    -0.17 |
| male         |    88.84 |

Модель была подогнана по 706 наблюдениям.



Интерпретируйте оценки коэффициентов при объясняющих переменных.

In [3]:
#| output: asis
b = res.params
display_answer(
    f'- Уровень–уровень. При увеличении времени работы (`totwrk`) на 1 минуту в неделю '
    f'при прочих равных время сна (`sleep`) в среднем {direction(b["totwrk"])} '
    f'на {fmt(abs(b["totwrk"]), 2)} минуты в неделю.\n'
    f'- Бинарная переменная. У мужчин (`male` = 1) при прочих равных время сна в среднем '
    f'на {fmt(abs(b["male"]), 2)} минуты в неделю {"больше" if b["male"] > 0 else "меньше"}, '
    f'чем у женщин.',
    theme='tip', title='Ответ')



::: {.callout-tip title="Ответ" icon="true"}
- Уровень–уровень. При увеличении времени работы (`totwrk`) на 1 минуту в неделю при прочих равных время сна (`sleep`) в среднем уменьшается на 0.17 минуты в неделю.
- Бинарная переменная. У мужчин (`male` = 1) при прочих равных время сна в среднем на 88.84 минуты в неделю больше, чем у женщин.
:::



## sleep equation #2

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ totwrk + male + smsa + age + south + yngkid + marr + union`

Объясняющие переменные: `totwrk` и `male`; остальные — контрольные.

Переменные: `smsa` = 1 для жителей крупной городской агломерации; `age` — возраст, лет; `south` = 1 для жителей Юга; `yngkid` = 1, если есть ребёнок младше 3 лет; `marr` = 1 для состоящих в браке; `union` = 1 для членов профсоюза.

Запишите спецификацию модели.

Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

Ответ:

In [4]:
#| output: asis
res = fit_ols('sleep ~ totwrk + male + smsa + age + south + yngkid + marr + union', sleep75)
show_coefs(res, decimals=2)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |  3446.83 |
| totwrk       |    -0.17 |
| male         |    87.11 |
| smsa         |   -54.19 |
| age          |     2.71 |
| south        |   102.27 |
| yngkid       |   -13.05 |
| marr         |    31.36 |
| union        |    11.87 |

Модель была подогнана по 706 наблюдениям.



Интерпретируйте оценки коэффициентов при объясняющих переменных.

# Полулогарифмические модели

## sleep equation #3

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ np.log(hrwage) + smsa + totwrk + male + marr + age + south + yngkid`

Объясняющие переменные: `np.log(hrwage)` и `smsa`; остальные — контрольные.

Переменные: `hrwage` — почасовая зарплата.

Запишите спецификацию модели.

Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

Ответ:

In [5]:
#| output: asis
res = fit_ols('sleep ~ np.log(hrwage) + smsa + totwrk + male + marr + age + south + yngkid',
              sleep75)
show_coefs(res, decimals=2)



| Переменная     |   Оценка |
|:---------------|---------:|
| Intercept      |  3440.19 |
| np.log(hrwage) |    -1.39 |
| smsa           |   -36.96 |
| totwrk         |    -0.16 |
| male           |    36.87 |
| marr           |    53.34 |
| age            |     2.37 |
| south          |    76.27 |
| yngkid         |    47.92 |

Модель была подогнана по 532 наблюдениям.



Интерпретируйте оценки коэффициентов при объясняющих переменных.

In [6]:
#| output: asis
b = res.params
display_answer(
    f'- Уровень–лог. При увеличении почасовой зарплаты (`hrwage`) на 1% при прочих равных '
    f'время сна (`sleep`) в среднем {direction(b["np.log(hrwage)"])} примерно '
    f'на {fmt(abs(b["np.log(hrwage)"]) / 100, 3)} минуты в неделю (оценка, делённая на 100).\n'
    f'- Бинарная переменная. У жителей крупной городской агломерации (`smsa` = 1) '
    f'при прочих равных время сна в среднем на {fmt(abs(b["smsa"]), 2)} минуты в неделю '
    f'{"больше" if b["smsa"] > 0 else "меньше"}, чем у остальных.',
    theme='tip', title='Ответ')



::: {.callout-tip title="Ответ" icon="true"}
- Уровень–лог. При увеличении почасовой зарплаты (`hrwage`) на 1% при прочих равных время сна (`sleep`) в среднем уменьшается примерно на 0.014 минуты в неделю (оценка, делённая на 100).
- Бинарная переменная. У жителей крупной городской агломерации (`smsa` = 1) при прочих равных время сна в среднем на 36.96 минуты в неделю меньше, чем у остальных.
:::



In [7]:
#| output: asis
display(Markdown(f'В наборе данных {len(sleep75)} наблюдений, а модель подогнана '
                 f'по {int(res.nobs)}. Чем это объясняется?'))
missing = sleep75['hrwage'].isna()
assert len(sleep75) - int(res.nobs) == missing.sum() and (sleep75.loc[missing, 'inlf'] == 0).all()
display_answer(
    f'У {missing.sum()} человек значение `hrwage` пропущено: все они не входят в состав '
    f'рабочей силы (`inlf` = 0). При подгонке statsmodels отбрасывает наблюдения с пропусками.',
    theme='tip', title='Ответ')

В наборе данных 706 наблюдений, а модель подогнана по 532. Чем это объясняется?



::: {.callout-tip title="Ответ" icon="true"}
У 174 человек значение `hrwage` пропущено: все они не входят в состав рабочей силы (`inlf` = 0). При подгонке statsmodels отбрасывает наблюдения с пропусками.
:::



## wage equation #1

Для набора данных `wage2` рассмотрим регрессию **np.log(wage) на age, IQ**.

Объясняющие переменные: `age` и `IQ`; контрольных нет.

Переменные: `wage` — месячная зарплата; `age` — возраст, лет; `IQ` — результат теста IQ, баллов.

Запишите спецификацию модели.

Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [8]:
#| output: asis
res = fit_ols('np.log(wage) ~ age + IQ', wage2)
show_coefs(res, decimals=3)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |    5.077 |
| age          |    0.024 |
| IQ           |    0.009 |

Модель была подогнана по 935 наблюдениям.



Интерпретируйте оценки коэффициентов при объясняющих переменных.

In [9]:
#| output: asis
b = res.params
display_answer(
    f'Лог–уровень.\n\n'
    f'- При увеличении возраста (`age`) на 1 год при прочих равных зарплата (`wage`) '
    f'в среднем {direction(b["age"])} примерно на {fmt(abs(b["age"]) * 100, 1)}% '
    f'(оценка, умноженная на 100).\n'
    f'- При увеличении `IQ` на 1 балл при прочих равных зарплата в среднем '
    f'{direction(b["IQ"])} примерно на {fmt(abs(b["IQ"]) * 100, 1)}%.',
    theme='tip', title='Ответ')



::: {.callout-tip title="Ответ" icon="true"}
Лог–уровень.

- При увеличении возраста (`age`) на 1 год при прочих равных зарплата (`wage`) в среднем увеличивается примерно на 2.4% (оценка, умноженная на 100).
- При увеличении `IQ` на 1 балл при прочих равных зарплата в среднем увеличивается примерно на 0.9%.
:::



## wage equation #2

Для набора данных `wage2` рассмотрим регрессию

`np.log(wage) ~ age + IQ + south + urban + married + KWW`

Объясняющие переменные: `age` и `IQ`; остальные — контрольные.

Переменные: `south` = 1 для жителей Юга; `urban` = 1 для жителей крупной городской агломерации; `married` = 1 для состоящих в браке; `KWW` — результат теста на знание рынка труда, баллов.

Запишите спецификацию модели.

Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [10]:
#| output: asis
res = fit_ols('np.log(wage) ~ age + IQ + south + urban + married + KWW', wage2)
show_coefs(res, decimals=3)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |    5.126 |
| age          |    0.014 |
| IQ           |    0.007 |
| south        |   -0.101 |
| urban        |    0.165 |
| married      |    0.191 |
| KWW          |    0.007 |

Модель была подогнана по 935 наблюдениям.



Интерпретируйте оценки коэффициентов при объясняющих переменных.

# Модели лог–лог

## output equation #1

Для набора данных `Labour` рассмотрим регрессию **np.log(output) на np.log(capital), np.log(labour)**.

Объясняющие переменные: `np.log(capital)` и `np.log(labour)`; контрольных нет.

Переменные: `output` — добавленная стоимость, млн евро; `capital` — основной капитал, млн евро; `labour` — число работников.

Запишите спецификацию модели.

Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [11]:
#| output: asis
res = fit_ols('np.log(output) ~ np.log(capital) + np.log(labour)', Labour)
show_coefs(res, decimals=3)



| Переменная      |   Оценка |
|:----------------|---------:|
| Intercept       |   -1.711 |
| np.log(capital) |    0.208 |
| np.log(labour)  |    0.715 |

Модель была подогнана по 569 наблюдениям.



Интерпретируйте оценки коэффициентов при объясняющих переменных.

In [12]:
#| output: asis
b = res.params
display_answer(
    f'Лог–лог.\n\n'
    f'- При увеличении капитала (`capital`) на 1% при прочих равных выпуск (`output`) '
    f'в среднем {direction(b["np.log(capital)"])} примерно '
    f'на {fmt(abs(b["np.log(capital)"]), 3)}%.\n'
    f'- При увеличении числа работников (`labour`) на 1% при прочих равных выпуск в среднем '
    f'{direction(b["np.log(labour)"])} примерно на {fmt(abs(b["np.log(labour)"]), 3)}%.',
    theme='tip', title='Ответ')



::: {.callout-tip title="Ответ" icon="true"}
Лог–лог.

- При увеличении капитала (`capital`) на 1% при прочих равных выпуск (`output`) в среднем увеличивается примерно на 0.208%.
- При увеличении числа работников (`labour`) на 1% при прочих равных выпуск в среднем увеличивается примерно на 0.715%.
:::



## output equation #2

Для набора данных `Labour` рассмотрим регрессию **np.log(output) на np.log(capital), np.log(labour), np.log(wage)**.

Объясняющие переменные: `np.log(capital)` и `np.log(labour)`; остальные — контрольные.

Переменные: `wage` — затраты на оплату труда на одного работника, тыс. евро.

Запишите спецификацию модели.

Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [13]:
#| output: asis
res = fit_ols('np.log(output) ~ np.log(capital) + np.log(labour) + np.log(wage)', Labour)
show_coefs(res, decimals=3)



| Переменная      |   Оценка |
|:----------------|---------:|
| Intercept       |   -5.007 |
| np.log(capital) |    0.149 |
| np.log(labour)  |    0.720 |
| np.log(wage)    |    0.921 |

Модель была подогнана по 569 наблюдениям.



Интерпретируйте оценки коэффициентов при объясняющих переменных.

## cost equation #1

Для набора данных `Electricity` рассмотрим регрессию **np.log(cost) на np.log(q)**.

Объясняющая переменная: `np.log(q)`; контрольных нет.

Переменные: `cost` — общие издержки; `q` — выпуск.

Запишите спецификацию модели.

Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [14]:
#| output: asis
res = fit_ols('np.log(cost) ~ np.log(q)', Electricity)
show_coefs(res, decimals=3)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |   -3.841 |
| np.log(q)    |    0.836 |

Модель была подогнана по 158 наблюдениям.



Интерпретируйте оценку коэффициента при объясняющей переменной.

## cost equation #2

Для набора данных `Electricity` рассмотрим регрессию **np.log(cost) на np.log(q), np.log(pl), np.log(pf), np.log(pk)**.

Объясняющая переменная: `np.log(q)`; остальные — контрольные.

Переменные: `pl` — ставка заработной платы; `pf` — цена топлива; `pk` — индекс цены капитала.

Запишите спецификацию модели.

Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [15]:
#| output: asis
res = fit_ols('np.log(cost) ~ np.log(q) + np.log(pl) + np.log(pf) + np.log(pk)', Electricity)
show_coefs(res, decimals=3)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |   -7.472 |
| np.log(q)    |    0.838 |
| np.log(pl)   |    0.044 |
| np.log(pf)   |    0.713 |
| np.log(pk)   |    0.188 |

Модель была подогнана по 158 наблюдениям.



Интерпретируйте оценку коэффициента при объясняющей переменной.